# 03 - PostgreSQL and PostGIS Loading

## Goal

Load the cleaned Ticketmaster event dataset into PostgreSQL and prepare the database for geospatial analysis with PostGIS.

## Tasks

- Connect to PostgreSQL
- Validate the PostGIS extension
- Create the event data model
- Load cleaned event data
- Create geographic points from coordinates
- Validate database records

In [3]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [9]:
project_path = Path("..")
load_dotenv(project_path / ".env", override= True)

db_user= os.getenv("POSTGRES_USER") 
db_password= os.getenv("POSTGRES_PASSWORD")
db_name= os.getenv("POSTGRES_DB")
db_port= os.getenv("POSTGRES_PORT")

In [10]:
assert db_user is not None
assert db_password is not None
assert db_name is not None
assert db_port is not None

In [15]:
database_url = (
    f"postgresql+psycopg2://"
    f"{db_user}:{db_password}"
    f"@localhost:{db_port}/{db_name}"
)

engine = create_engine(database_url)


In [16]:
with engine.connect() as connection:
    result= connection.execute(
        text(
            "SELECT 1;"
        )
    )
    print(result.scalar())

1


## Database Connection

The PostgreSQL connection is validated before executing the project schema.

In [69]:
with engine.connect() as connection:
    postgis_version = connection.execute(
        text("SELECT PostGIS_Version();")
    ).scalar()

postgis_version

'3.5 USE_GEOS=1 USE_PROJ=1 USE_STATS=1'

In [74]:
schema_path = (
    project_path
    / "sql"
    / "01_create_events_table.sql"
)

schema_sql = schema_path.read_text(encoding="utf-8")

print("Schema file exists:", schema_path.exists())
print("Schema length:", len(schema_sql))

Schema file exists: True
Schema length: 634


In [75]:
raw_connection = engine.raw_connection()

try:
    cursor = raw_connection.cursor()
    cursor.execute(schema_sql)
    raw_connection.commit()
finally:
    cursor.close()
    raw_connection.close()

In [76]:
with engine.connect() as connection:
    location_exists = connection.execute(
        text("""
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.columns
                WHERE table_name = 'events'
                  AND column_name = 'location'
            );
        """)
    ).scalar()

location_exists

True

In [77]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE table_name = 'events'
            );
        """)
    )

    events_table_exists = result.scalar()

events_table_exists

True

In [78]:
print("Rows:", len(events_df))
print("Duplicate event IDs:", events_df["event_id"].duplicated().sum())
print("Unique event IDs:", events_df["event_id"].is_unique)

print("\nRequired fields:")
print("Missing event IDs:", events_df["event_id"].isna().sum())
print("Missing event dates:", events_df["event_date"].isna().sum())
print("Missing cities:", events_df["city"].isna().sum())
print("Missing countries:", events_df["country"].isna().sum())
print("Missing latitude:", events_df["latitude"].isna().sum())
print("Missing longitude:", events_df["longitude"].isna().sum())

Rows: 1172
Duplicate event IDs: 0
Unique event IDs: True

Required fields:
Missing event IDs: 0
Missing event dates: 0
Missing cities: 0
Missing countries: 0
Missing latitude: 0
Missing longitude: 0


## Load Cleaned Event Data

The cleaned Ticketmaster dataset is loaded from the processed data directory before being inserted into PostgreSQL.

In [80]:
clean_file = (project_path/ "data" / "processed" / "ticketmaster_events_clean.csv")

events_df = pd.read_csv(clean_file,
                        parse_dates=["event_date"])

In [81]:
events_df.head()

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
0,LvZ18QLUFcKuwNYZ0XXWn,Heavysaurus - METAL Tour 2026,Heavysaurus,2026-08-30,13:00:00,Schön & Frölich,Braunschweig,Germany,52.25654,10.49957,https://www.universe.com/events/heavysaurus-me...
1,Z698xZC2Z16v8KeAfJ,Melanie Martinez – HADES: THE SACRIFICE | VIP,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
2,Z698xZC2Z1kAIGIZg,Melanie Martinez – HADES: THE SACRIFICE,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
3,LvZ18Qpz8oKu0POZyE61A,SUPERBLOOM 2026 Experience Day - Sonntag,SUPERBLOOM Festival,2026-08-30,10:00:00,NaN,Munich,Germany,48.17429,11.55524,https://www.universe.com/events/superbloom-202...
4,Z698xZC2Z1kCpjv8P,ITZY 3RD WORLD TOUR <TUNNEL VISION> in FRANKFURT,ITZY,2026-09-17,19:30:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/itzy-3rd-wor...


In [82]:
events_df.shape

(1172, 11)

In [83]:
events_df.dtypes

event_id                  str
event_name                str
artist_name               str
event_date     datetime64[us]
event_time                str
venue_name                str
city                      str
country                   str
latitude              float64
longitude             float64
event_url                 str
dtype: object

In [85]:
print("Rows:", len(events_df))
print("Duplicate event IDs:", events_df["event_id"].duplicated().sum())
print("Unique event IDs:", events_df["event_id"].is_unique)

Rows: 1172
Duplicate event IDs: 0
Unique event IDs: True


In [86]:
with engine.begin() as connection:
    connection.execute(
        text(
            "TRUNCATE TABLE events;"
        )
    )

In [87]:
events_df.to_sql(
    name= "events",
    con= engine,
    if_exists="replace",
    index=False,
    method= "multi"
    )

c:\Users\annav\miniconda3\envs\gigroute_env\Lib\site-packages\pandas\io\sql.py:2078: SAWarning: Did not recognize type 'geography' of column 'location'
  self.meta.reflect(


1172

In [88]:
with engine.connect() as connection:
    database_count = connection.execute(
        text(
            "SELECT COUNT(*) FROM events"
        )
    ).scalar()

print("DataFrame rows:", len(events_df))
print("Database rows:", database_count)

DataFrame rows: 1172
Database rows: 1172


In [89]:
with engine.connect() as connection:
    duplicate_count = connection.execute(
        text(
            """
                SELECT COUNT(*)
            FROM (
                SELECT event_id
                FROM events
                GROUP BY event_id
                HAVING COUNT(*) > 1
            ) duplicates;
            """
        )
    ).scalar()
print("Duplicate event IDs in database:", duplicate_count)

Duplicate event IDs in database: 0


In [91]:
with engine.begin() as connection:
    connection.execute(
        text("""
            ALTER TABLE events
            ADD COLUMN IF NOT EXISTS location GEOGRAPHY(POINT, 4326);
        """)
    )

In [92]:
with engine.connect() as connection:
    location_exists = connection.execute(
        text("""
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.columns
                WHERE table_name = 'events'
                  AND column_name = 'location'
            );
        """)
    ).scalar()

location_exists

True

In [93]:
with engine.begin() as connection:
    connection.execute(
        text("""
            CREATE INDEX IF NOT EXISTS idx_events_location
            ON events
            USING GIST (location);
        """)
    )

In [94]:
with engine.begin() as connection:
    connection.execute(
        text("""
            UPDATE events
            SET location = ST_SetSRID(
                ST_MakePoint(longitude, latitude),
                4326
            )::geography;
        """)
    )

In [95]:
with engine.connect() as connection:
    missing_locations = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM events
            WHERE location IS NULL;
        """)
    ).scalar()

print("Missing PostGIS locations:", missing_locations)

Missing PostGIS locations: 0
